# Vector Databases for RAG

> **From an in-memory prototype to a real searchable vector store with Qdrant Cloud.**

In the previous tutorial, we built a simple semantic retriever.

We generated embeddings for our documents, embedded a user's query, calculated similarity scores, and returned the top results.

The architecture looked like this:

```text
User Query
    ↓
Query Embedding
    ↓
Similarity Search
    ↓
Ranked Results
```

But our implementation had an obvious limitation.

We kept all of the vectors in Python memory.

That works for a handful of chunks.

It is not how we would build a serious knowledge base.

In this tutorial, we'll introduce **vector databases** and move our RAG system to **Qdrant Cloud**.

By the end, our retrieval architecture will look like:

```text
Documents
    ↓
Embeddings
    ↓
Qdrant Cloud
    ↓
Vector Search
    ↓
Retrieved Chunks
```

The goal is not simply to learn how to call the Qdrant API.

The goal is to understand **what a vector database contributes to a RAG system and why it exists**.

## What We'll Learn

By the end of this tutorial, you'll understand:

- Why our in-memory vector search does not scale
- What a vector database actually stores
- The difference between vectors and metadata
- What a Qdrant collection is
- What a Qdrant point is
- Vector dimensions
- Distance metrics
- Creating a collection
- Uploading vectors and payloads
- Searching Qdrant
- Filtering results using metadata
- Updating and deleting indexed information
- Connecting Qdrant Cloud to our existing RAG pipeline

We'll use **Qdrant Cloud** throughout this tutorial so that the notebook works in both local JupyterLab and Google Colab.

---
# 1. Why Do We Need a Vector Database?

In the previous notebook, our retrieval code was essentially:

```python
scores = document_embeddings @ query_embedding
```

We then sorted those scores and selected the top results.

That is perfectly fine for a small experiment.

But imagine:

```text
10 chunks
100 chunks
1,000 chunks
10,000 chunks
1,000,000 chunks
```

Keeping everything in Python memory and calculating every comparison ourselves becomes increasingly impractical.

We therefore want infrastructure designed for:

- Storing vectors
- Searching vectors efficiently
- Storing associated metadata
- Filtering records
- Updating records
- Deleting records
- Managing larger collections

That's where a vector database comes in.

---
# 2. What Is a Vector Database?

A vector database is a system designed to store and retrieve vector representations efficiently.

In our RAG system, a stored record conceptually looks like:

```text
┌──────────────────────────────────────┐
│               POINT                  │
├──────────────────────────────────────┤
│ ID                                   │
│ Vector                               │
│ Payload / Metadata                   │
└──────────────────────────────────────┘
```

For example:

```python
{
    "id": "refund-policy-001",
    "vector": [0.021, -0.184, 0.731, ...],
    "payload": {
        "text": "Customers can request...",
        "source": "refund_policy.pdf",
        "page": 4
    }
}
```

The vector is useful for similarity search.

The payload gives us the information we actually want to retrieve and use.

This distinction will be important throughout the course.

---
# 3. Qdrant Cloud

For this course, we'll use [**Qdrant Cloud**](https://qdrant.tech/)  rather than running Qdrant locally.

This has an important advantage:

> **The same notebook can be used by someone running JupyterLab locally or someone running it in Google Colab.**

You need a [Qdrant Cloud account](https://qdrant.tech/) and an API key.

Create a [Qdrant Cloud account](https://qdrant.tech/) and create a cluster.

You will need two values:

```text
QDRANT_URL
QDRANT_API_KEY
```

Keep the API key private.

**Never put the key directly into a notebook that you intend to publish or commit to GitHub.**

## Storing Credentials Safely

For local JupyterLab, you can store your Qdrant credentials in a `.env` file.

For example:

```bash
QDRANT_URL="your-qdrant-url"
QDRANT_API_KEY="your-api-key"
```

For Google Colab, use **Colab Secrets** to store the values instead of writing them directly into a code cell.

The notebook will read both values through Python:

In [2]:
! pip install -q python-dotenv


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")

If you're using Colab, retrieve the values from your secrets store and expose them to the notebook environment.

The important principle is:

```text
Secret
  ↓
Environment / Secret Store
  ↓
Python
  ↓
Qdrant Client
```

not:

```text
Secret
  ↓
Notebook code
  ↓
GitHub
```

# 4. Install the Dependencies

We'll install the client directly from the notebook.

In [5]:
! pip install -q qdrant-client sentence-transformers 


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


---
# 5. Import the Libraries

In [6]:
import os

from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

## 6. Load the Embedding Model

We'll use `BAAI/bge-small-en-v1.5` as our embedding model.

In [8]:
embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# 7. Create Our Knowledge Base

We'll use a small set of documents so that we can focus on how the vector database works.

In [9]:
documents = [
    {
        "id": "refund_policy",
        "text": "Customers can request a refund within 30 days of purchase."
    },
    {
        "id": "refund_processing",
        "text": "Approved refunds are normally processed within 7 business days."
    },
    {
        "id": "refund_condition",
        "text": "Products must be returned in their original condition to qualify for a refund."
    },
    {
        "id": "shipping_standard",
        "text": "Standard shipping normally takes between 3 and 5 business days."
    },
    {
        "id": "shipping_express",
        "text": "Express shipping normally takes between 1 and 2 business days."
    },
    {
        "id": "shipping_tracking",
        "text": "Customers can track orders using the tracking number provided by email."
    },
    {
        "id": "support_hours",
        "text": "Customer support is available Monday through Friday from 9 AM to 5 PM."
    }
]

# 8. Generate Document Embeddings

Now we explicitly create `document_embeddings`.

In [10]:
texts = [document["text"] for document in documents]

document_embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True
)

The resulting array has the conceptual shape:

```text
(number_of_documents, embedding_dimension)
```

Let's inspect it.

In [11]:
print(f"Embedding shape: {document_embeddings.shape}")

Embedding shape: (7, 384)


## 9. Connect to Qdrant CLoud

In [12]:
client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
)

Now test the connection:

In [13]:
client.get_collections()

CollectionsResponse(collections=[])

If the connection succeeds, Qdrant will return the collections available in your account.

We've now moved from:

```text
Python memory
```

to:

```text
Qdrant Cloud
```

as the storage layer for our vector data.

# 10. Collections

A **collection** is a named group of points that share a vector configuration.

Conceptually:

```text
Qdrant Cloud
     │
     ├── customer-support
     ├── product-docs
     └── research-docs
```

For this tutorial, we'll use:

```text
rag-tutorial
```

# 11. Vector Dimensions

We already generated our embeddings.

Let's retrieve their dimensionality:

```python
vector_size = document_embeddings.shape[1]
print(vector_size)
```

The collection must use this same dimension.

> **The vector size configured in Qdrant must match the dimension produced by the embedding model.**

In [14]:
vector_size = document_embeddings.shape[1]

print(f"Vector dimension: {vector_size}")

Vector dimension: 384


# 12. Distance Metrics

A vector database needs to know how vectors should be compared.

Common choices include:

- Cosine similarity
- Dot product
- Euclidean distance

We'll use **cosine distance** in Qdrant for this tutorial.

The important relationship is:

```text
Embedding Model
       +
Distance Metric
       ↓
Retrieval Behavior
```

# 13. Create the Collection

In [28]:
collection_name = "rag-tutorial"

# Delete the collection if it already exists
if client.collection_exists(collection_name):
    client.delete_collection(collection_name)

client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=vector_size,
        distance=Distance.COSINE,
    ),
)

True

We now have an empty collection ready to receive points.

# 14. What Is a Point?

In Qdrant, our records are stored as **points**.

A point combines:

```text
Point
│
├── ID
├── Vector
└── Payload
      ├── text
      ├── document_id
      └── other metadata
```

This maps naturally to the records created during indexing.

# 15. Build Qdrant Points

We'll use `PointStruct` to represent each record.


In [29]:
points = []

for point_id, (document, embedding) in enumerate(zip(documents, document_embeddings)):
    points.append(
        PointStruct(
            id=point_id,
            vector=embedding.tolist(),
            payload={
                "text": document["text"],
                "document_id": document["id"],
            },
        )
    )

The vector is the searchable representation.

The payload stores the information we want to retrieve alongside it.

# 16. Upload the Points

Send the points to Qdrant Cloud:

In [30]:
# Create a point id mapping
point_id_map = {
    document["id"]: index
    for index, document in enumerate(documents)
}

point_id_map

{'refund_policy': 0,
 'refund_processing': 1,
 'refund_condition': 2,
 'shipping_standard': 3,
 'shipping_express': 4,
 'shipping_tracking': 5,
 'support_hours': 6}

In [31]:
client.upsert(
    collection_name=collection_name,
    points=points,
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

`upsert` inserts a point or updates an existing point with the same ID.

Our architecture is now:

```text
Documents
    ↓
Embeddings
    ↓
PointStruct
    ↓
Qdrant Cloud
```

Let's verify the collection:

In [32]:
client.get_collection(collection_name)

CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, warnings=None, indexed_vectors_count=0, points_count=7, segments_count=2, config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, memory=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, payload=None, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, memory=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_

Our vector data is no longer only inside the notebook's Python process.

It lives in our Qdrant Cloud collection.

# 17. Search the Vector Database

Now let's answer:

> **How long do I have to request a refund?**

Create the query embedding:

In [33]:
query = "How long do I have to request a refund?"

query_embedding = embedding_model.encode(
    query,
    normalize_embeddings=True
)

Search Qdrant:

In [34]:
search_results = client.query_points(
    collection_name=collection_name,
    query=query_embedding.tolist(),
    limit=3,
).points

for result in search_results:
    print(f"Score: {result.score:.4f}")
    print(f"ID: {result.id}")
    print(result.payload["text"])
    print("---")

Score: 0.8186
ID: 0
Customers can request a refund within 30 days of purchase.
---
Score: 0.8045
ID: 1
Approved refunds are normally processed within 7 business days.
---
Score: 0.7495
ID: 2
Products must be returned in their original condition to qualify for a refund.
---


# 18. What Just Happened?

Previously:

```text
Query
  ↓
Embedding
  ↓
NumPy
  ↓
Compare vectors
```

Now:

```text
Query
  ↓
Embedding
  ↓
Qdrant Cloud
  ↓
Vector Search
  ↓
Top-K Points
```

The conceptual retrieval operation has not changed.

What changed is the **storage and search infrastructure**.

> **A vector database does not replace retrieval. It provides infrastructure for storing and searching vectors.**

# 19. Payloads Are More Than Text

So far our payload contains:

```python
{
    "text": "...",
    "document_id": "..."
}
```

A real application may store:

```python
{
    "text": "...",
    "document_id": "refund-policy",
    "source": "refund_policy.pdf",
    "page": 4,
    "section": "Refund Eligibility",
    "version": 3,
    "department": "customer-service"
}
```

This information can support:

- Citations
- Filtering
- Access control
- Debugging
- Document versioning
- Provenance

The vector tells us about the position in embedding space.

The payload tells us what the vector represents.

# 20. Metadata Filtering

Vector similarity isn't the only way we may want to search.

Suppose our knowledge base contains documents from multiple departments.

We might want:

> **Search only customer-service documents.**

This is where metadata filtering becomes useful.

Conceptually:

```text
Query
  ↓
Vector Search
  +
Metadata Filter
  ↓
Relevant Results
```

We'll explore richer metadata filters as our knowledge base becomes more realistic.

# 21. Why Filtering Matters

Filtering can affect correctness, not just performance.

Imagine a company has:

```text
Public policy
Internal policy
Archived policy
Customer-specific policy
```

A user should not necessarily retrieve all of them.

Without appropriate filtering, a system could retrieve information that:

- Belongs to another tenant
- Is outdated
- Is outside the user's permissions
- Applies to a different product
- Comes from the wrong region

Metadata and retrieval design can therefore become part of the application's security model.

# 22. Updating a Point

Documents change.

Suppose the refund policy changes from:

```text
30 days
```

to:

```text
60 days
```

Create a new embedding:

In [35]:
updated_text = (
    "Customers can request a refund within 60 days of purchase."
)

updated_embedding = embedding_model.encode(
    updated_text,
    normalize_embeddings=True
)

client.upsert(
    collection_name=collection_name,
    points=[
        PointStruct(
            id=point_id_map["refund_policy"],  # Qdrant point ID for refund_policy
            vector=updated_embedding.tolist(),
            payload={
                "text": updated_text,
                "document_id": "refund_policy",
            },
        )
    ],
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

The old representation has now been replaced for that point ID.

> **Changing source content generally means changing its embedding too.**

# 23. Deleting a Point

If a document should no longer be searchable, remove its indexed representation.

In [36]:
client.delete(
    collection_name=collection_name,
    points_selector=[point_id_map["refund_policy"]],
)

UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

Document lifecycle therefore becomes:

```text
Create
  ↓
Index
  ↓
Update
  ↓
Re-index
  ↓
Delete / Archive
```

# 24. Connecting Qdrant to Our RAG Pipeline

We can now connect everything we've learned.

### Indexing

```text
Documents
    ↓
Chunking
    ↓
Embeddings
    ↓
Qdrant Cloud
```

### Querying

```text
User Query
    ↓
Query Embedding
    ↓
Qdrant Search
    ↓
Retrieved Points
    ↓
Payload Text
    ↓
Context
    ↓
LLM
```

The complete system is:

```text
                    INDEXING
                        │
                        ▼
                    Documents
                        │
                        ▼
                    Chunking
                        │
                        ▼
                    Embeddings
                        │
                        ▼
                  ┌───────────┐
                  │  Qdrant   │
                  │   Cloud   │
                  └─────┬─────┘
                        │
                     QUERY
                        │
                        ▼
                    User Query
                        │
                        ▼
                 Query Embedding
                        │
                        ▼
                  Vector Search
                        │
                        ▼
                 Retrieved Points
                        │
                        ▼
                     Context
                        │
                        ▼
                       LLM
                        │
                        ▼
                     Answer
```

# 25. Vector Database vs RAG

It's important not to confuse these concepts.

A vector database is **not** a RAG system.

RAG is the larger architecture:

```text
RAG
│
├── Document ingestion
├── Chunking
├── Embeddings
├── Indexing
├── Retrieval
├── Context construction
└── Generation
```

Qdrant is one component inside that architecture:

```text
RAG System
     │
     └── Vector Database
             │
             └── Qdrant
```

Keeping these concepts separate prevents a lot of confusion.

# 26. What a Vector Database Solves

Our original prototype had to manage:

```text
Embeddings in Python
Similarity calculations
Metadata
Storage
Updates
Deletes
```

Qdrant gives us infrastructure for these storage and retrieval concerns.

The larger RAG system still has to solve:

```text
Document ingestion
Chunking
Embedding selection
Query processing
Retrieval strategy
Reranking
Context construction
Generation
Evaluation
```

So introducing Qdrant doesn't magically make our RAG system production-ready.

It gives us a proper foundation for one important part of it: **vector storage and search**.

# Key Takeaways

1. A vector database stores and searches vector representations efficiently.
2. Our in-memory NumPy retriever is useful for learning but does not provide scalable storage infrastructure.
3. Qdrant Cloud lets us use a real vector database from both local JupyterLab and Google Colab.
4. A Qdrant point combines an ID, vector, and optional payload.
5. The vector is used for similarity search.
6. Payloads preserve text and metadata associated with the vector.
7. The collection's vector dimension must match the embedding dimension.
8. The distance metric determines how vectors are compared.
9. Metadata filtering can constrain retrieval to the appropriate subset of knowledge.
10. Updating source content generally requires updating its embedding.
11. A vector database is **one component of a RAG system**, not the RAG system itself.

The mental model to keep is:

```text
                  RAG
                   │
        ┌──────────┴──────────┐
        ↓                     ↓
    Indexing                Querying
        │                     │
        ↓                     ↓
   Embeddings              Retrieval
        │                     │
        ↓                     ↓
   Qdrant Cloud          Retrieved Points
                              │
                              ↓
                           Context
                              │
                              ↓
                             LLM
                              │
                              ↓
                           Answer
```

# What's Next?

We now have a proper vector store.

But our retrieval system still has a significant limitation:

```text
Query
  ↓
Dense Embedding
  ↓
Vector Search
```

What happens when the user searches for an exact product code?

Or a policy number?

Or a person's name?

Or a technical identifier?

Pure semantic retrieval isn't always enough.

The next step is to combine different retrieval signals:

```text
                 Query
                   │
          ┌────────┴────────┐
          ↓                 ↓
    Dense Retrieval    Keyword Retrieval
          ↓                 ↓
       Candidates        Candidates
          └────────┬────────┘
                   ↓
            Combined Ranking
```

In the next notebook, we'll explore:

> **Hybrid Retrieval — combining semantic and keyword search.**

We'll see why dense and lexical retrieval complement each other and how techniques such as **Reciprocal Rank Fusion (RRF)** can combine their results.